In [ ]:
from qdts_nn.model import *
from qdts_nn.train import *
from qdts_nn.auxiliar import *

import torch
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================
# Main Training Loop
# ==============================
def train_multiple_runs(
    n,
    hidden_size=128,
    num_layers=3,
    batch_size=512,
    lr=1e-3,
    num_repeats=10,
    weight_reg_one=0.0,
    save_path = "models",
    device=None,
 ):
    # make directory if it doesn't exist
    import os
    os.makedirs(save_path, exist_ok=True)

    all_runs_results = []

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # print(f"    🚀 Training {num_repeats} runs | n={n}\n")

    # 🔒 Hardcoded curriculum
    radii  = [0.1, 0.2, 0.3, 0.4, 0.5]
    epochs = 20
    batches_per_epoch = 100
    weight_reg_zero   = 0.01
    weight_reg_one    = weight_reg_one

    for run_idx in range(1, num_repeats+1):
        seed = run_idx * 123
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        # print(f"    🔃 Training new model")

        model = model_creator(
            n=n,
            hidden_size=hidden_size,
            num_layers=num_layers,
        ).to(device)

        run_history = []

        # 🚀 Curriculum loop (no config, just values)
        for r in tqdm(radii, leave=False):
            model, results = train_model(
                model=model,
                epochs=epochs,
                batches_per_epoch=batches_per_epoch,
                batch_size=batch_size,
                n=n,
                data_radius=r,
                lr=lr,
                weight_reg_one=weight_reg_one,
                weight_reg_zero=weight_reg_zero,
                device=device,
            )

            run_history.extend(results["loss_history"])
            
        print(f"    ✅ Run finished")

        # 💾 Save
        save_path = f"{save_path}/Model_N{n}_L{num_layers}_H{hidden_size}_R{weight_reg_one}_seed{seed}.pt"
        save_model(model, save_path, n, hidden_size, num_layers)

        # 🔄 Reload (your workflow)
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        model = load_model(save_path, device=device)

        # 📊 Validation
        val = validate_model(model, n, 2048, 0.5, device=device)

        print(f"    📊 Validation (regular/clamped):")
        print(f"        loss_inverse_distortion: {val['loss_inverse_distortion']:.3e} / {val['loss_inverse_distortion_clamped']:.3e}")
        print(f"        reg_one:                 {val['reg_one']:.3e} / {val['reg_one_clamped']:.3e}")
        print(f"        reg_zero:                {val['reg_zero']:.3e} / {val['reg_zero_clamped']:.3e}")

        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        print(f"    💾 Saved: {save_path}")

        all_runs_results.append({
            "N": n,
            "seed": seed,
            "weight_reg_one": weight_reg_one,
            "val_loss_inverse_distortion": val['loss_inverse_distortion'],
            "val_loss_inverse_distortion_clamped": val['loss_inverse_distortion_clamped'],
            "val_reg_one": val['reg_one'],
            "val_reg_one_clamped": val['reg_one_clamped'],
            "val_reg_zero": val['reg_zero'],
            "val_reg_zero_clamped": val['reg_zero_clamped'],
        })

    return all_runs_results

Using device: cuda


In [2]:
all_results = []
for n in range(5, 17):
    print(f"\n=== Training for N={n} ===")
    results = train_multiple_runs(
        n=n,
        hidden_size=256,
        num_layers=3,
        batch_size=256,
        lr=1e-3,
        num_repeats=8,
        weight_reg_one=0.0,
        device=device,
    )
    all_results.extend(results)
    results = train_multiple_runs(
        n=n,
        hidden_size=256,
        num_layers=3,
        batch_size=256,
        lr=1e-3,
        num_repeats=8,
        weight_reg_one=0.25,
        device=device,
    )
    all_results.extend(results)

import csv

with open("results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
    writer.writeheader()
    writer.writerows(all_results)


=== Training for N=5 ===


    ✅ Run finished


RuntimeError: Parent directory models does not exist.